# 06 · HPO temporal sin fuga de información

Este notebook optimiza hiperparámetros mediante validación temporal expansiva. XGBoost es el modelo principal; Random Forest y la regresión logística actúan como comparadores, y LightGBM se incorpora si está instalado.

La búsqueda solo utiliza filas etiquetadas con `dataset_split == 'train'`. Octubre, noviembre y diciembre de 2022 quedan excluidos incluso de la lista de archivos leídos. El notebook no consulta validation ni test.

## Protocolo metodológico

Se utilizan cuatro folds expansivos:

1. Train: 2019; validación interna: enero–marzo de 2021.
2. Train: 2019 y enero–junio de 2021; validación interna: julio–septiembre de 2021.
3. Train: 2019–2021; validación interna: enero–marzo de 2022.
4. Train: 2019–2021 y enero–junio de 2022; validación interna: julio–septiembre de 2022.

Las fronteras se aplican simultáneamente a todas las estaciones. Se purga una hora al final de train porque la etiqueta se define a `t+1`. Los transformadores se ajustan desde cero únicamente con el train de cada fold.

In [ ]:
# %pip install scikit-learn xgboost optuna matplotlib
# %pip install lightgbm

from pathlib import Path
import gc
import warnings

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
from IPython.display import display
from optuna.samplers import NSGAIISampler
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

# LightGBM es opcional: la ausencia del paquete no detiene el resto del HPO.
try:
    from lightgbm import LGBMClassifier
    LIGHTGBM_AVAILABLE = True
except ImportError:
    LIGHTGBM_AVAILABLE = False
    print('LightGBM no está instalado; se omite su búsqueda opcional.')

warnings.filterwarnings('ignore', category=FutureWarning)
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Localizamos las carpetas del proyecto desde la raíz o desde notebooks/.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'Datos modelado').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
FEATURES_DIR = PROJECT_ROOT / 'Datos modelado' / 'estacion_hora_features'
HPO_DIR = PROJECT_ROOT / 'Datos modelado' / 'hpo_temporal'
HPO_DIR.mkdir(parents=True, exist_ok=True)

TARGET = 'risk_class_1h'
TIME_COLUMN = 'fecha_hora_local'
RANDOM_STATE = 42
HORIZON_HOURS = 1
CHUNK_SIZE = 100_000
SEEDS_STABILITY = [13, 42, 97]

# Límites prácticos para la fase de búsqueda. Pueden aumentarse si hay más memoria y tiempo.
MAX_TRAIN_ROWS_PER_FOLD = 250_000
MAX_VALIDATION_ROWS_PER_FOLD = 100_000

# Número de configuraciones que evaluará cada familia.
N_TRIALS = {
    'xgboost': 80,
    'random_forest': 40,
    'logistic': 24,
    'lightgbm': 40,
}
RUN_LIGHTGBM = LIGHTGBM_AVAILABLE


## Variables predictivas y exclusión física de octubre–diciembre

La selección de archivos admite solo 2019, 2021 y enero–septiembre de 2022. Después se vuelve a exigir `dataset_split == 'train'` y etiqueta no nula. Esta doble barrera impide consultar accidentalmente los conjuntos externos.

In [ ]:
NUMERIC_FEATURES = [
    'capacity', 'bikes_available', 'docks_available', 'reservations_count',
    'occupancy_ratio', 'light', 'weather_available',
    'uv_radiation_median_mw_m2', 'wind_speed_median_m_s',
    'wind_direction_sin_mean', 'wind_direction_cos_mean',
    'temperature_median_c', 'relative_humidity_median_pct',
    'barometric_pressure_median_mb', 'solar_radiation_median_w_m2',
    'precipitation_mean_l_m2', 'precipitation_max_l_m2',
    'n_temperature', 'n_relative_humidity', 'n_precipitation',
    'hour', 'day_of_week', 'month', 'week_of_year',
    'occupancy_ratio_lag_1h', 'occupancy_ratio_lag_2h', 'occupancy_ratio_lag_24h',
    'bikes_available_lag_1h', 'bikes_available_lag_2h', 'bikes_available_lag_24h',
    'net_flow_lag_1h', 'net_flow_lag_2h', 'net_flow_lag_24h',
    'departures_count_lag_1h', 'departures_count_lag_2h', 'departures_count_lag_24h',
    'arrivals_count_lag_1h', 'arrivals_count_lag_2h', 'arrivals_count_lag_24h',
    'occupancy_ratio_mean_previous_3h', 'occupancy_ratio_mean_previous_24h',
    'net_flow_mean_previous_3h', 'net_flow_mean_previous_24h',
    'departures_count_mean_previous_3h', 'departures_count_mean_previous_24h',
    'arrivals_count_mean_previous_3h', 'arrivals_count_mean_previous_24h',
]
CATEGORICAL_FEATURES = ['station_id', 'tipo_dia']
MODEL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
READ_COLUMNS = MODEL_FEATURES + [TARGET, 'dataset_split', TIME_COLUMN]

def file_year_month(file_path: Path) -> tuple[int, int]:
    """Extrae AAAA y MM del nombre estacion_hora_features_AAAAMM.csv."""
    token = file_path.stem.rsplit('_', maxsplit=1)[-1]
    return int(token[:4]), int(token[4:6])

all_feature_files = sorted(FEATURES_DIR.glob('estacion_hora_features_*.csv'))
feature_files = []
for file_path in all_feature_files:
    year, month = file_year_month(file_path)
    if year in {2019, 2021} or (year == 2022 and month <= 9):
        feature_files.append(file_path)

if not feature_files:
    raise FileNotFoundError(f'No se encontraron particiones de train en {FEATURES_DIR}')

# Auditoría: ningún archivo de octubre, noviembre o diciembre de 2022 puede superar este punto.
assert all(not (file_year_month(path)[0] == 2022 and file_year_month(path)[1] >= 10) for path in feature_files)
print(f'Archivos disponibles en la carpeta: {len(all_feature_files)}')
print(f'Archivos autorizados para HPO: {len(feature_files)}')
print('Último archivo autorizado:', feature_files[-1].name)


## Definición de folds temporales

Las fechas de inicio y fin son límites globales para todas las estaciones. `validation_end` es exclusivo. Las filas de train cuya etiqueta futura alcanzaría el comienzo de validación se eliminan mediante la purga de una hora.

In [ ]:
FOLD_SPECS = [
    {'fold': 'fold_1', 'validation_start': '2021-01-01 00:00:00', 'validation_end': '2021-04-01 00:00:00'},
    {'fold': 'fold_2', 'validation_start': '2021-07-01 00:00:00', 'validation_end': '2021-10-01 00:00:00'},
    {'fold': 'fold_3', 'validation_start': '2022-01-01 00:00:00', 'validation_end': '2022-04-01 00:00:00'},
    {'fold': 'fold_4', 'validation_start': '2022-07-01 00:00:00', 'validation_end': '2022-10-01 00:00:00'},
]

for spec in FOLD_SPECS:
    spec['validation_start'] = pd.Timestamp(spec['validation_start'])
    spec['validation_end'] = pd.Timestamp(spec['validation_end'])
    spec['train_end_exclusive'] = spec['validation_start'] - pd.Timedelta(hours=HORIZON_HOURS)

def parse_local_hour(values: pd.Series) -> pd.Series:
    """Convierte la fecha local ignorando el sufijo horario, para comparar límites de calendario."""
    return pd.to_datetime(values.astype('string').str.slice(0, 19), errors='coerce')

fold_definition = pd.DataFrame(FOLD_SPECS)[
    ['fold', 'train_end_exclusive', 'validation_start', 'validation_end']
]
display(fold_definition)


## Conteo y muestreo estratificado por fold

Primero se cuentan las clases de cada ventana leyendo solo fecha, etiqueta y split. Después se carga una muestra proporcional de cada clase. El muestreo reduce el coste, pero nunca mezcla periodos: la pertenencia temporal se decide antes de seleccionar filas.

In [ ]:
def empty_class_counts() -> dict:
    return {
        spec['fold']: {part: {0: 0, 1: 0, 2: 0} for part in ('train', 'validation')}
        for spec in FOLD_SPECS
    }

def temporal_masks(local_hour: pd.Series, spec: dict) -> tuple[pd.Series, pd.Series]:
    """Crea máscaras disjuntas aplicadas igual a todas las estaciones."""
    train_mask = local_hour.lt(spec['train_end_exclusive'])
    validation_mask = local_hour.ge(spec['validation_start']) & local_hour.lt(spec['validation_end'])
    return train_mask, validation_mask

fold_class_counts = empty_class_counts()
for file_path in feature_files:
    for chunk in pd.read_csv(
        file_path, usecols=[TIME_COLUMN, TARGET, 'dataset_split'],
        chunksize=CHUNK_SIZE, low_memory=False,
    ):
        # Primera condición obligatoria: solo split train y etiqueta observada.
        chunk = chunk.loc[chunk['dataset_split'].eq('train') & chunk[TARGET].notna()].copy()
        if chunk.empty:
            continue
        local_hour = parse_local_hour(chunk[TIME_COLUMN])
        for spec in FOLD_SPECS:
            train_mask, validation_mask = temporal_masks(local_hour, spec)
            for part_name, mask in [('train', train_mask), ('validation', validation_mask)]:
                counts = chunk.loc[mask, TARGET].astype(int).value_counts()
                for class_value, count in counts.items():
                    fold_class_counts[spec['fold']][part_name][int(class_value)] += int(count)

count_rows = []
for fold_name, part_counts in fold_class_counts.items():
    for part_name, class_counts in part_counts.items():
        for class_value, count in class_counts.items():
            count_rows.append({
                'fold': fold_name, 'part': part_name,
                'risk_class': class_value, 'rows_available': count,
            })
fold_counts_table = pd.DataFrame(count_rows)
display(fold_counts_table)


In [ ]:
def class_sampling_probabilities(class_counts: dict, maximum_rows: int) -> dict:
    """Conserva aproximadamente la distribución original de las tres clases."""
    total = sum(class_counts.values())
    if total <= maximum_rows:
        return {class_value: 1.0 for class_value in class_counts}
    probabilities = {}
    for class_value, count in class_counts.items():
        desired = maximum_rows * count / total
        probabilities[class_value] = min(1.0, desired / count) if count else 0.0
    return probabilities

sampling_probabilities = {}
rng_by_group = {}
for fold_index, spec in enumerate(FOLD_SPECS):
    fold_name = spec['fold']
    for part_index, part_name in enumerate(('train', 'validation')):
        limit = MAX_TRAIN_ROWS_PER_FOLD if part_name == 'train' else MAX_VALIDATION_ROWS_PER_FOLD
        sampling_probabilities[(fold_name, part_name)] = class_sampling_probabilities(
            fold_class_counts[fold_name][part_name], limit
        )
        for class_value in (0, 1, 2):
            # Semillas fijas y distintas para que cada muestra sea reproducible.
            seed = RANDOM_STATE + 100 * fold_index + 10 * part_index + class_value
            rng_by_group[(fold_name, part_name, class_value)] = np.random.default_rng(seed)

sample_parts = {
    spec['fold']: {'train': [], 'validation': []} for spec in FOLD_SPECS
}

for file_path in feature_files:
    for chunk in pd.read_csv(file_path, usecols=READ_COLUMNS, chunksize=CHUNK_SIZE, low_memory=False):
        chunk = chunk.loc[chunk['dataset_split'].eq('train') & chunk[TARGET].notna()].copy()
        if chunk.empty:
            continue
        chunk['__local_hour'] = parse_local_hour(chunk[TIME_COLUMN])
        chunk[TARGET] = chunk[TARGET].astype('int8')

        for spec in FOLD_SPECS:
            fold_name = spec['fold']
            train_mask, validation_mask = temporal_masks(chunk['__local_hour'], spec)
            for part_name, temporal_mask in [('train', train_mask), ('validation', validation_mask)]:
                candidate = chunk.loc[temporal_mask]
                if candidate.empty:
                    continue
                selected_positions = []
                for class_value in (0, 1, 2):
                    class_positions = np.flatnonzero(candidate[TARGET].to_numpy() == class_value)
                    if len(class_positions) == 0:
                        continue
                    probability = sampling_probabilities[(fold_name, part_name)][class_value]
                    rng = rng_by_group[(fold_name, part_name, class_value)]
                    selected_positions.append(class_positions[rng.random(len(class_positions)) < probability])
                if selected_positions:
                    selected_positions = np.concatenate(selected_positions)
                    if len(selected_positions):
                        sample_parts[fold_name][part_name].append(candidate.iloc[selected_positions].copy())

raw_folds = []
for spec in FOLD_SPECS:
    fold_name = spec['fold']
    train_frame = pd.concat(sample_parts[fold_name]['train'], ignore_index=True)
    validation_frame = pd.concat(sample_parts[fold_name]['validation'], ignore_index=True)

    # Comprobaciones que detienen el proceso ante cualquier fuga temporal o de split.
    assert train_frame['dataset_split'].eq('train').all()
    assert validation_frame['dataset_split'].eq('train').all()
    assert train_frame['__local_hour'].max() < validation_frame['__local_hour'].min()
    assert set(train_frame['__local_hour'].unique()).isdisjoint(set(validation_frame['__local_hour'].unique()))

    raw_folds.append({
        'fold': fold_name, 'train': train_frame, 'validation': validation_frame,
    })
    print(
        fold_name,
        '| train:', f'{len(train_frame):,}',
        '| validation:', f'{len(validation_frame):,}',
        '| última hora train:', train_frame['__local_hour'].max(),
        '| primera hora validation:', validation_frame['__local_hour'].min(),
    )

del sample_parts
gc.collect()


## Preprocesamiento independiente dentro de cada fold

Cada fold construye un imputador y un codificador nuevos. `fit_transform` se aplica únicamente a su train; la validación interna pasa solo por `transform`. Las matrices resultantes pueden reutilizarse entre trials porque el preprocesamiento no depende de los hiperparámetros del modelo.

In [ ]:
def make_preprocessor() -> ColumnTransformer:
    """Devuelve un transformador sin ajustar para utilizarlo en un único fold."""
    numeric_pipeline = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
    ])
    categorical_pipeline = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('one_hot', OneHotEncoder(handle_unknown='ignore')),
    ])
    return ColumnTransformer(transformers=[
        ('numeric', numeric_pipeline, NUMERIC_FEATURES),
        ('categorical', categorical_pipeline, CATEGORICAL_FEATURES),
    ])

prepared_folds = []
for raw_fold in raw_folds:
    preprocessor = make_preprocessor()
    train_frame = raw_fold['train']
    validation_frame = raw_fold['validation']

    # Único ajuste permitido para este fold.
    X_train = preprocessor.fit_transform(train_frame[MODEL_FEATURES])
    # La validación no modifica medianas ni categorías aprendidas.
    X_validation = preprocessor.transform(validation_frame[MODEL_FEATURES])

    prepared_folds.append({
        'fold': raw_fold['fold'],
        'X_train': X_train,
        'y_train': train_frame[TARGET].astype(int).reset_index(drop=True),
        'X_validation': X_validation,
        'y_validation': validation_frame[TARGET].astype(int).reset_index(drop=True),
    })
    print(raw_fold['fold'], '| matriz train:', X_train.shape, '| matriz validation:', X_validation.shape)

# Los DataFrames originales ya no son necesarios durante la búsqueda.
del raw_folds
gc.collect()


## Espacios de búsqueda

XGBoost explora profundidad, regularización, muestreo de filas/columnas y pesos de riesgo. Random Forest busca complejidad, tamaño de hoja y balanceo. La logística recibe un ajuste pequeño de regularización. LightGBM, si está disponible, explora una región comparable a XGBoost.

In [ ]:
def suggest_parameters(model_key: str, trial: optuna.Trial) -> dict:
    """Define únicamente parámetros que pueden aprenderse dentro del HPO."""
    if model_key == 'xgboost':
        return {
            'n_estimators': trial.suggest_int('n_estimators', 300, 1400, step=100),
            'max_depth': trial.suggest_int('max_depth', 3, 10),
            'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.20, log=True),
            'min_child_weight': trial.suggest_float('min_child_weight', 1.0, 20.0, log=True),
            'subsample': trial.suggest_float('subsample', 0.60, 1.00),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.60, 1.00),
            'gamma': trial.suggest_float('gamma', 0.0, 5.0),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 30.0, log=True),
            'balanced_classes': trial.suggest_categorical('balanced_classes', [False, True]),
            'weight_empty': trial.suggest_float('weight_empty', 0.70, 2.50),
            'weight_full': trial.suggest_float('weight_full', 0.70, 2.50),
        }
    if model_key == 'random_forest':
        return {
            'n_estimators': trial.suggest_int('n_estimators', 300, 800, step=100),
            'max_depth': trial.suggest_categorical('max_depth', [None, 12, 18, 24, 32]),
            'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 20),
            'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', 0.4, 0.7]),
            'max_samples': trial.suggest_float('max_samples', 0.60, 1.00),
            'class_weight': trial.suggest_categorical('class_weight', [None, 'balanced', 'balanced_subsample']),
        }
    if model_key == 'logistic':
        return {
            'C': trial.suggest_float('C', 1e-3, 20.0, log=True),
            'penalty': trial.suggest_categorical('penalty', ['l1', 'l2']),
            'class_weight': trial.suggest_categorical('class_weight', [None, 'balanced']),
        }
    if model_key == 'lightgbm':
        return {
            'n_estimators': trial.suggest_int('n_estimators', 300, 1200, step=100),
            'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.20, log=True),
            'num_leaves': trial.suggest_int('num_leaves', 15, 127),
            'max_depth': trial.suggest_categorical('max_depth', [-1, 5, 8, 12, 16]),
            'min_child_samples': trial.suggest_int('min_child_samples', 10, 120),
            'subsample': trial.suggest_float('subsample', 0.60, 1.00),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.60, 1.00),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 30.0, log=True),
            'class_weight': trial.suggest_categorical('class_weight', [None, 'balanced']),
        }
    raise ValueError(f'Modelo desconocido: {model_key}')


## Evaluación multobjetivo

Cada trial devuelve dos objetivos: media de F1 macro y media de balanced accuracy. También guarda sus desviaciones entre folds. El orden de selección posterior será: F1 macro primero; dentro de una tolerancia de 0,005, balanced accuracy; y finalmente menor desviación de F1.

In [ ]:
def make_model(model_key: str, parameters: dict, seed: int):
    """Construye un modelo nuevo para cada fold y evita compartir estado aprendido."""
    if model_key == 'xgboost':
        model_parameters = {
            key: value for key, value in parameters.items()
            if key not in {'balanced_classes', 'weight_empty', 'weight_full'}
        }
        return XGBClassifier(
            objective='multi:softprob', num_class=3, eval_metric='mlogloss',
            tree_method='hist', early_stopping_rounds=50,
            n_jobs=-1, random_state=seed, **model_parameters,
        )
    if model_key == 'random_forest':
        return RandomForestClassifier(n_jobs=-1, random_state=seed, **parameters)
    if model_key == 'logistic':
        return LogisticRegression(
            solver='saga', max_iter=400, n_jobs=-1, random_state=seed, **parameters,
        )
    if model_key == 'lightgbm':
        return LGBMClassifier(
            objective='multiclass', num_class=3, subsample_freq=1,
            n_jobs=-1, random_state=seed, verbosity=-1, **parameters,
        )
    raise ValueError(f'Modelo desconocido: {model_key}')

def xgboost_weights(y_train: pd.Series, parameters: dict) -> np.ndarray:
    """Combina balanceo automático y multiplicadores específicos de vaciado/saturación."""
    if parameters['balanced_classes']:
        weights = compute_sample_weight(class_weight='balanced', y=y_train).astype(float)
    else:
        weights = np.ones(len(y_train), dtype=float)
    y_array = y_train.to_numpy()
    weights[y_array == 1] *= parameters['weight_empty']
    weights[y_array == 2] *= parameters['weight_full']
    return weights

def fit_and_predict(model_key: str, parameters: dict, fold: dict, seed: int) -> np.ndarray:
    """Entrena desde cero en un fold y devuelve predicciones de su validación interna."""
    model = make_model(model_key, parameters, seed)
    if model_key == 'xgboost':
        model.fit(
            fold['X_train'], fold['y_train'],
            sample_weight=xgboost_weights(fold['y_train'], parameters),
            eval_set=[(fold['X_validation'], fold['y_validation'])],
            verbose=False,
        )
    else:
        model.fit(fold['X_train'], fold['y_train'])
    return model.predict(fold['X_validation']).astype(int)

def evaluate_parameters(model_key: str, parameters: dict, seed: int) -> pd.DataFrame:
    """Calcula F1 macro y balanced accuracy en los cuatro folds temporales."""
    rows = []
    for fold in prepared_folds:
        prediction = fit_and_predict(model_key, parameters, fold, seed)
        rows.append({
            'model': model_key,
            'seed': seed,
            'fold': fold['fold'],
            'f1_macro': f1_score(fold['y_validation'], prediction, average='macro'),
            'balanced_accuracy': balanced_accuracy_score(fold['y_validation'], prediction),
        })
        del prediction
        gc.collect()
    return pd.DataFrame(rows)

def objective_for(model_key: str):
    """Crea la función objetivo multobjetivo del modelo solicitado."""
    def objective(trial: optuna.Trial):
        parameters = suggest_parameters(model_key, trial)
        fold_metrics = evaluate_parameters(model_key, parameters, RANDOM_STATE)
        f1_mean = fold_metrics['f1_macro'].mean()
        balanced_mean = fold_metrics['balanced_accuracy'].mean()
        trial.set_user_attr('f1_std', float(fold_metrics['f1_macro'].std(ddof=0)))
        trial.set_user_attr('balanced_accuracy_std', float(fold_metrics['balanced_accuracy'].std(ddof=0)))
        trial.set_user_attr('fold_f1_macro', fold_metrics['f1_macro'].round(8).tolist())
        trial.set_user_attr('fold_balanced_accuracy', fold_metrics['balanced_accuracy'].round(8).tolist())
        return float(f1_mean), float(balanced_mean)
    return objective


## Ejecución de Optuna

Los estudios se guardan en SQLite para poder reanudar la búsqueda. Se usa un trial cada vez porque cada estimador ya paraleliza internamente; lanzar varios trials simultáneos podría saturar memoria y CPU.

In [ ]:
model_keys = ['xgboost', 'random_forest', 'logistic']
if RUN_LIGHTGBM:
    model_keys.append('lightgbm')

storage_url = f"sqlite:///{(HPO_DIR / 'optuna_hpo_temporal.db').as_posix()}"
studies = {}
for model_key in model_keys:
    print(f'Iniciando/continuando HPO de {model_key}...')
    study = optuna.create_study(
        study_name=f'hpo_temporal_{model_key}',
        directions=['maximize', 'maximize'],
        sampler=NSGAIISampler(seed=RANDOM_STATE),
        storage=storage_url,
        load_if_exists=True,
    )
    study.optimize(
        objective_for(model_key),
        n_trials=N_TRIALS[model_key],
        n_jobs=1,
        gc_after_trial=True,
        show_progress_bar=True,
    )
    studies[model_key] = study
    print(f'{model_key}: {len(study.trials)} trials acumulados')


## Selección jerárquica y estabilidad

No se mezclan arbitrariamente las dos métricas. Primero se conserva cualquier configuración a menos de 0,005 del mejor F1 macro de su familia. Entre ellas se escoge la mayor balanced accuracy y, si persiste el empate, la menor desviación temporal de F1.

In [ ]:
F1_TOLERANCE = 0.005

def study_results_table(model_key: str, study: optuna.Study) -> pd.DataFrame:
    """Convierte los trials completados en una tabla CSV fácil de auditar."""
    rows = []
    for trial in study.trials:
        if trial.state != optuna.trial.TrialState.COMPLETE or trial.values is None:
            continue
        row = {
            'model': model_key,
            'trial_number': trial.number,
            'f1_macro_mean': trial.values[0],
            'balanced_accuracy_mean': trial.values[1],
            'f1_macro_std': trial.user_attrs.get('f1_std', np.nan),
            'balanced_accuracy_std': trial.user_attrs.get('balanced_accuracy_std', np.nan),
        }
        row.update({f'param_{key}': value for key, value in trial.params.items()})
        rows.append(row)
    return pd.DataFrame(rows)

def select_trial(table: pd.DataFrame) -> pd.Series:
    """Aplica F1 primero, balanced accuracy después y estabilidad como desempate."""
    best_f1 = table['f1_macro_mean'].max()
    candidates = table.loc[table['f1_macro_mean'].ge(best_f1 - F1_TOLERANCE)].copy()
    return candidates.sort_values(
        ['balanced_accuracy_mean', 'f1_macro_std', 'f1_macro_mean'],
        ascending=[False, True, False],
    ).iloc[0]

trial_tables = {}
selected_trials = {}
selected_parameters = {}
selection_rows = []
for model_key, study in studies.items():
    table = study_results_table(model_key, study)
    table.to_csv(HPO_DIR / f'trials_{model_key}.csv', index=False, encoding='utf-8-sig')
    selected_row = select_trial(table)
    selected_trial_number = int(selected_row['trial_number'])
    selected_trial = next(trial for trial in study.trials if trial.number == selected_trial_number)

    trial_tables[model_key] = table
    selected_trials[model_key] = selected_trial
    selected_parameters[model_key] = selected_trial.params
    selection_rows.append(selected_row.to_dict())

selection_summary = pd.DataFrame(selection_rows).sort_values('f1_macro_mean', ascending=False)
selection_summary.to_csv(HPO_DIR / 'configuraciones_seleccionadas_hpo.csv', index=False, encoding='utf-8-sig')
display(selection_summary[[
    'model', 'trial_number', 'f1_macro_mean', 'balanced_accuracy_mean',
    'f1_macro_std', 'balanced_accuracy_std',
]])


In [ ]:
# Repetimos cada configuración ganadora con tres semillas para medir estabilidad algorítmica.
stability_parts = []
for model_key, parameters in selected_parameters.items():
    print(f'Repitiendo {model_key} con semillas {SEEDS_STABILITY}...')
    for seed in SEEDS_STABILITY:
        stability_parts.append(evaluate_parameters(model_key, parameters, seed))

stability_by_fold = pd.concat(stability_parts, ignore_index=True)
stability_summary = (
    stability_by_fold.groupby('model', as_index=False)
    .agg(
        f1_macro_mean=('f1_macro', 'mean'),
        f1_macro_std=('f1_macro', 'std'),
        balanced_accuracy_mean=('balanced_accuracy', 'mean'),
        balanced_accuracy_std=('balanced_accuracy', 'std'),
    )
    .sort_values('f1_macro_mean', ascending=False)
)
stability_by_fold.to_csv(HPO_DIR / 'estabilidad_por_semilla_y_fold.csv', index=False, encoding='utf-8-sig')
stability_summary.to_csv(HPO_DIR / 'resumen_estabilidad_hpo.csv', index=False, encoding='utf-8-sig')
display(stability_summary)


## Visualización de compromisos y exportación de parámetros

El gráfico permite ver el frente entre F1 macro y balanced accuracy. El CSV de parámetros guarda una fila por parámetro y modelo, evitando depender del fichero interno de Optuna para documentar la configuración elegida.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for model_key, table in trial_tables.items():
    ax.scatter(
        table['balanced_accuracy_mean'], table['f1_macro_mean'],
        alpha=0.45, s=24, label=model_key,
    )
ax.set_title('HPO temporal: F1 macro frente a balanced accuracy')
ax.set_xlabel('Balanced accuracy media')
ax.set_ylabel('F1 macro medio')
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()

parameter_rows = []
for model_key, parameters in selected_parameters.items():
    for parameter, value in parameters.items():
        parameter_rows.append({
            'model': model_key, 'parameter': parameter, 'value': value,
        })
selected_parameters_table = pd.DataFrame(parameter_rows)
selected_parameters_table.to_csv(HPO_DIR / 'mejores_hiperparametros.csv', index=False, encoding='utf-8-sig')
display(selected_parameters_table)


## Auditoría final y uso correcto

Este notebook termina sin abrir octubre, noviembre o diciembre de 2022 y sin entrenar un modelo contra test. XGBoost sigue siendo la familia final; el HPO determina su configuración dentro de train. Los demás modelos son comparadores de sensibilidad.

Como noviembre–diciembre ya se evaluó antes de realizar este HPO, no debe reutilizarse para escoger entre configuraciones. En el TFM conviene presentar los resultados temporales internos y sus desviaciones, mantener el test anterior como referencia congelada y documentar esta limitación.

In [ ]:
# Generamos una auditoría sencilla de las barreras contra fuga utilizadas.
audit_rows = [
    {'control': 'splits_permitidos', 'value': 'train'},
    {'control': 'ultimo_mes_leido', 'value': feature_files[-1].stem.rsplit('_', 1)[-1]},
    {'control': 'horizonte_objetivo_horas', 'value': HORIZON_HOURS},
    {'control': 'n_folds_temporales', 'value': len(FOLD_SPECS)},
    {'control': 'test_consultado', 'value': False},
]
audit_table = pd.DataFrame(audit_rows)
audit_table.to_csv(HPO_DIR / 'auditoria_sin_fuga.csv', index=False, encoding='utf-8-sig')
display(audit_table)
